# Phase 4 — ELO Model + Monte Carlo Simulation

**Goal:** Estimate Portugal's probability of winning the 2026 FIFA World Cup using ELO ratings and 100,000 Monte Carlo simulations.

### Bracket structure
- 48 teams → 12 groups of 4
- **32 advance:** Top 2 from each group (24) + 8 best 3rd-place teams (8)
- Round of 32 → R16 → QF → SF → Final
- Sections: Groups A-D (Sec 1), E-H (Sec 2), **I-L (Sec 3 — Portugal)**, best-8 3rd-place (Sec 4)

### ELO formula
$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$

In [1]:
import sys
sys.path.insert(0, '..')   # so 'src' is importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from src.elo import elo_win_prob, match_probs
from src.simulation import (
    load_team_data, load_groups,
    run_monte_carlo, run_portugal_path_analysis
)

CHARTS = Path('../outputs/charts')
SIM    = Path('../simulation')
CHARTS.mkdir(parents=True, exist_ok=True)
SIM.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans'})
print('Ready.')

Ready.


## 1. Team strengths — ELO snapshot

In [2]:
elo, lam = load_team_data()
groups_df = load_groups()

# All WC 2026 teams with ELO
wc_teams = groups_df['team'].tolist()
wc_elo = pd.DataFrame([
    {'team': t, 'group': groups_df.loc[groups_df['team']==t, 'group'].values[0],
     'elo': elo.get(t, 1500)}
    for t in wc_teams
]).sort_values('elo', ascending=False).reset_index(drop=True)

print('Top 15 teams by ELO (WC 2026):')
print(wc_elo.head(15).to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\team_features.csv'

In [ ]:
# Group K detail
group_k = wc_elo[wc_elo['group'] == 'K']
print('Group K — Portugal\'s group:')
print(group_k.to_string(index=False))

# Portugal vs Colombia — key group match
p_port_win = elo_win_prob(elo['Portugal'], elo['Colombia'])
p_col_win  = elo_win_prob(elo['Colombia'], elo['Portugal'])
pw, pd_, pl = match_probs(elo['Portugal'], elo['Colombia'])
print(f"\nPortugal vs Colombia:  Win={pw:.1%}  Draw={pd_:.1%}  Loss={pl:.1%}")

## 2. Run 100,000 Monte Carlo simulations

In [ ]:
%%time
N_SIMS = 100_000
results = run_monte_carlo(N_SIMS, seed=42)

results.to_csv(SIM / 'elo_simulation_results.csv', index=False)
print(f'Simulations complete. Unique champions: {len(results)}')
print()
print('Top 15 champions:')
print(results.head(15).to_string(index=False))

In [ ]:
port_row = results[results['winner'] == 'Portugal']
port_prob = port_row['probability'].values[0] if len(port_row) else 0.0
print(f"=== Portugal WC 2026 win probability (ELO model): {port_prob:.2%} ===")

## 3. Portugal path analysis

In [ ]:
%%time
path = run_portugal_path_analysis(N_SIMS, seed=42)

path_df = pd.DataFrame([
    {'stage': s, 'count': c, 'probability': c / N_SIMS}
    for s, c in path.items()
])
print('Portugal stage-by-stage breakdown:')
print(path_df.to_string(index=False))

## 4. Charts

In [ ]:
# Chart 1 — Top 10 champion probabilities
top10 = results.head(10).copy()
colors = ['#006600' if t == 'Portugal' else '#2c7bb6' for t in top10['winner']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top10['winner'][::-1], top10['probability'][::-1], color=colors[::-1])
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('Win probability')
ax.set_title('2026 FIFA World Cup — Championship probability (ELO model, 100k simulations)')
for bar, p in zip(bars, top10['probability'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{p:.1%}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(CHARTS / '08_elo_champion_probs.png')
plt.show()
print('Saved: 08_elo_champion_probs.png')

In [ ]:
# Chart 2 — Portugal path funnel
stages = list(path.keys())
probs  = [path[s] / N_SIMS for s in stages]

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = ['#d73027' if s != 'Winner' else '#1a9850' for s in stages]
ax.bar(stages, probs, color=bar_colors, edgecolor='white')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylabel('Probability')
ax.set_title('Portugal — elimination stage distribution (ELO model)')
for i, (s, p) in enumerate(zip(stages, probs)):
    ax.text(i, p + 0.003, f'{p:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(CHARTS / '09_portugal_path_elo.png')
plt.show()
print('Saved: 09_portugal_path_elo.png')

In [ ]:
# Chart 3 — ELO distribution for all 48 WC teams
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(wc_elo['elo'], bins=20, color='#4393c3', edgecolor='white')
ax.axvline(elo.get('Portugal', 1500), color='green', linewidth=2, label=f"Portugal ({elo.get('Portugal', 1500):.0f})")
ax.axvline(elo.get('Spain', 1500), color='red', linewidth=2, linestyle='--', label=f"Spain ({elo.get('Spain', 1500):.0f})")
ax.set_xlabel('ELO rating')
ax.set_ylabel('Number of teams')
ax.set_title('ELO rating distribution — 2026 World Cup teams')
ax.legend()
plt.tight_layout()
plt.savefig(CHARTS / '10_elo_distribution_wc26.png')
plt.show()
print('Saved: 10_elo_distribution_wc26.png')

## 5. Summary

| Metric | Value |
|---|---|
| Model | ELO-based Monte Carlo |
| Simulations | 100,000 |
| Portugal ELO | 1,976 (8th globally) |
| Portugal group | K (Uzbekistan, Colombia, DR Congo) |
| **Portugal WC win probability** | **see cell above** |

**Key limitations of this model:**
- Bracket structure is an approximation — actual FIFA 2026 draw positions differ slightly
- Static ELO (no in-tournament updating)
- Does not capture form, injuries, or squad depth

**Next:** Phase 5 — Dixon-Coles model (Poisson-based, captures scoreline probabilities)